# **Kernel SHAP estimators: Practice**

Hello, everyone. In the lesson “How the speed-ups and the computation of SHAP work” we saw how Shapley values are computed from a sample of coalitions instead of enumerating all $2^n$ of them. Kernel SHAP ([Lundberg and Lee, 2017](https://arxiv.org/abs/1705.07874)) finds Shapley values as the solution of a weighted linear regression: coalitions are drawn with Shapley kernel weights, and the regression coefficients of the feature indicators are the $\phi_i$. In the notation of [Covert and Lee, 2021](https://arxiv.org/abs/2012.01536), the solution of this regression under the efficiency constraint is

$$\phi = A^{-1}\Big(b - \mathbf{1}\,\frac{\mathbf{1}^\top A^{-1} b - \big(v(N) - v(\varnothing)\big)}{\mathbf{1}^\top A^{-1}\mathbf{1}}\Big),\qquad A = \mathbb{E}_z\big[zz^\top\big],\quad b = \mathbb{E}_z\big[z\,\big(v(z) - v(\varnothing)\big)\big],$$

where $z$ is a random coalition written as a vector of zeros and ones, and the expectation is taken over the Shapley kernel distribution. The matrix $A$ depends only on the number of features and is known in closed form, while $b$ has to be estimated from a sample of coalitions. The original Kernel SHAP estimates both $A$ and $b$ from the sample, so it is slightly biased. Unbiased KernelSHAP takes the exact $A$ and estimates only $b$: the estimate of $\phi$ is linear in $b$ and therefore unbiased.

Today we will see what this difference gives in practice, on the familiar diabetes dataset and a random forest of 300 trees; the same dataset and model come back later, in the lesson “SAGE: Practice”. Diabetes has 10 features, so the exact answer can be computed by enumerating all 1024 coalitions, and the error of each estimator can be measured against it.

In this practice you will:
- compute exact Shapley values for one test row with `shapiq.ExactComputer` and check them against `shap.TreeExplainer`;
- run `shap.KernelExplainer`, `shapiq.KernelSHAP` and `shapiq.UnbiasedKernelSHAP` at budgets from 64 to 1024 value function evaluations, 30 seeds per budget, and split the error of each estimator into bias and variance;
- build confidence intervals for Unbiased KernelSHAP with the `shapley-regression` package and check how often they cover the exact value;
- see what the estimators do not fix: the choice of background changes the exact values themselves.

**Happy coding!**

As always, we start by installing the dependencies. Colab lacks `shapiq` and `shapley-regression`; we pin the versions so that the numbers match ours. `shap` 0.52.0 and `scikit-learn` are already on Colab.

In [ ]:
!pip install shapiq==1.7.0 shapley-regression==0.0.1 -q

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import shapiq
from shapreg import removal, games, shapley, shapley_unbiased

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# some numpy builds print false overflow warnings from matrix multiplication; they do not affect the numbers
warnings.filterwarnings('ignore', message='.*encountered in matmul')

## Data, model and game

The dataset and the model are the same as those in the later lesson “SAGE: Practice”. The diabetes dataset is built into `sklearn`: 442 patients, 10 features (age, sex, body mass index `bmi`, blood pressure `bp` and six blood measurements `s1`–`s6`), and the target is a quantitative measure of disease progression one year later. 30% of the rows are held out for the test, and the model is a random forest of 300 trees. We will explain the prediction for the first test row.

In [ ]:
X, y = load_diabetes(return_X_y=True, as_frame=True)
feature_names = list(X.columns)
n_players = X.shape[1]

X_train, X_test, y_train, y_test = train_test_split(X.values, y.values, test_size=0.3, random_state=0)
model = RandomForestRegressor(n_estimators=300, max_features=0.33, min_samples_leaf=3,
                              random_state=0, n_jobs=-1).fit(X_train, y_train)

x = X_test[0]
print(f'prediction for the first test row: {model.predict(x[None])[0]:.2f}, true value: {y_test[0]:.0f}')

`shap.KernelExplainer` plays the interventional (marginal) game: the features in a coalition $S$ are taken from the explained row $x$, the rest from the rows of the background $B$, and the predictions are averaged over the background:

$$v(S) = \frac{1}{|B|}\sum_{r\in B} f\big(x_S,\, r_{\bar S}\big).$$

The background $B$ is 64 random rows of the training set. Let us write this game as a function that takes a whole batch of coalitions at once (a boolean matrix of shape (number of coalitions, 10)) and counts how many values $v(S)$ were computed. This count is what we will call the budget of an estimator.

In [ ]:
background = X_train[np.random.default_rng(0).choice(len(X_train), 64, replace=False)]


class Game:
    def __init__(self, model, x, background):
        self.model, self.x, self.background = model, x, background
        self.calls = 0                                         # how many values v(S) have been computed

    def __call__(self, coalitions):
        coalitions = np.asarray(coalitions, dtype=bool)
        self.calls += len(coalitions)
        # features in S come from x, the rest from each background row
        rows = np.where(coalitions[:, None, :], self.x[None, None, :], self.background[None, :, :])
        predictions = self.model.predict(rows.reshape(-1, n_players))
        return predictions.reshape(len(coalitions), len(self.background)).mean(axis=1)


game = Game(model, x, background)
v_empty = game(np.zeros((1, n_players)))[0]
v_full = game(np.ones((1, n_players)))[0]
print(f'v(empty coalition) = {v_empty:.2f}: the mean prediction over the background')
print(f'v(N) = {v_full:.2f}: the prediction for x; the features share v(N) - v(empty) = {v_full - v_empty:.2f}')

## Exact values: `shapiq.ExactComputer`

With 10 features there are $2^{10} = 1024$ coalitions, so exact Shapley values can be computed by full enumeration. `shapiq.ExactComputer` evaluates the game on all coalitions once and computes any index from this table; the index `'SV'` is the Shapley values.

In [ ]:
game.calls = 0
exact = shapiq.ExactComputer(game, n_players=n_players)('SV')
phi_exact = np.array([exact[(i,)] for i in range(n_players)])

print(f'values v(S) computed: {game.calls}')
print(pd.Series(phi_exact, index=feature_names).round(3).sort_values(key=abs, ascending=False).to_string())
print(f'\nsum of phi = {phi_exact.sum():.4f}, v(N) - v(empty) = {v_full - v_empty:.4f}')

Trees offer a second route to the same numbers. `shap.TreeExplainer` with `feature_perturbation='interventional'` and the same background computes the Shapley values of the same game by traversing the trees, without enumerating coalitions. With `feature_perturbation='tree_path_dependent'` and no background, it plays a different game: a missing feature is marginalized over the shares of training rows that went into each branch of a tree, not over the background. Let us compare all three columns.

In [ ]:
tree_interventional = shap.TreeExplainer(model, data=background,
                                         feature_perturbation='interventional').shap_values(x[None])[0]
tree_path = shap.TreeExplainer(model, feature_perturbation='tree_path_dependent').shap_values(x[None])[0]

trees = pd.DataFrame({'ExactComputer': phi_exact, 'Tree interventional': tree_interventional,
                      'Tree path-dependent': tree_path}, index=feature_names)
print(trees.round(3).to_string())
print(f'\nmax |ExactComputer - interventional| = {np.abs(phi_exact - tree_interventional).max():.4f}, '
      f'sum of interventional - (v(N) - v(empty)) = {tree_interventional.sum() - (v_full - v_empty):.4f}')
print(f'max |ExactComputer - path-dependent| = {np.abs(phi_exact - tree_path).max():.4f}')

**Self-check question.** Which feature got the largest exact Shapley value in absolute terms, and what is it? How far apart are `ExactComputer` and the interventional `TreeExplainer`, and why is the gap larger for path-dependent?

<details><summary>Check yourself</summary>

The largest exact value in absolute terms belongs to `bmi`: 48.178, followed by `bp` (25.641) and `s3` (10.939). The sum of the ten values, 95.2309, equals $v(N)-v(\varnothing) = 243.63-148.40$: efficiency holds to the last printed digit. The interventional `TreeExplainer` differs from `ExactComputer` by at most 0.038 (for `s3`: 10.901 against 10.939), and the deviation comes from `TreeExplainer` itself: the sum of its values differs from $v(N)-v(\varnothing)$ by 0.0104. The path-dependent `TreeExplainer` differs by up to 4.159 (for `s5`: −2.017 against 2.143) because it computes the Shapley values of a different game: it marginalizes a missing feature over the training rows in the branches of the trees, not over the 64 background rows. Neither method is wrong here; they answer two different questions.

</details>

## Three estimators at different budgets

Now let us forget that we have the exact answer and give the estimators a limited budget, from 64 to 1024 evaluations of $v(S)$.

- **`shap.KernelExplainer(f, background)`** plays the same game, and `nsamples` is how many coalitions it will evaluate. It is thriftier than the textbook algorithm: it first fully enumerates the coalitions of the smallest and largest sizes (1 and $n-1$, then 2 and $n-2$ and so on) as long as the budget allows, and fills the rest with random coalitions, each together with its complement. `l1_reg=False` turns off feature selection before the regression: with 10 features it drops nothing anyway.
- **`shapiq.KernelSHAP`** is the original Kernel SHAP: both $A$ and $b$ are estimated from the sample of coalitions.
- **`shapiq.UnbiasedKernelSHAP`** is the unbiased variant of Covert and Lee: $A$ is exact, and only $b$ is estimated from the sample.

Both `shapiq` estimators have a `pairing_trick` parameter: when it is on, each coalition is drawn together with its complement; these are the pairs of complementary coalitions from the lesson “How the speed-ups and the computation of SHAP work”. We check the `shapiq` budget with the game counter. `KernelExplainer` calls the model itself, so we wrap the model to count rows and divide that number by the size of the background.

In [ ]:
budgets = [64, 128, 256, 512, 1024]
n_seeds = 30


class CountingModel:
    def __init__(self, model):
        self.model, self.rows = model, 0

    def __call__(self, rows):
        self.rows += len(rows)
        return self.model.predict(rows)


def kernel_explainer(budget, seed):
    np.random.seed(seed)                                       # KernelExplainer draws random coalitions from np.random
    f = CountingModel(model)
    explainer = shap.KernelExplainer(f, background)
    f.rows = 0                                                 # do not count the calls made while building the explainer
    values = explainer.shap_values(x[None], nsamples=budget, l1_reg=False, silent=True)[0]
    return values, f.rows / len(background)


def shapiq_estimator(approximator_class, pairing):
    def estimate(budget, seed):
        approximator = approximator_class(n=n_players, pairing_trick=pairing, random_state=seed)
        values = approximator.approximate(budget, game)
        return np.array([values[(i,)] for i in range(n_players)])
    return estimate


estimators = {
    'shap.KernelExplainer': lambda budget, seed: kernel_explainer(budget, seed)[0],
    'shapiq.KernelSHAP': shapiq_estimator(shapiq.KernelSHAP, pairing=False),
    'shapiq.KernelSHAP, pairs': shapiq_estimator(shapiq.KernelSHAP, pairing=True),
    'shapiq.UnbiasedKernelSHAP': shapiq_estimator(shapiq.UnbiasedKernelSHAP, pairing=False),
    'shapiq.UnbiasedKernelSHAP, pairs': shapiq_estimator(shapiq.UnbiasedKernelSHAP, pairing=True),
}

for budget in budgets:
    game.calls = 0
    estimators['shapiq.UnbiasedKernelSHAP'](budget, 0)
    print(f'budget {budget:>4}: KernelExplainer computed {kernel_explainer(budget, 0)[1]:.0f} values v(S), '
          f'UnbiasedKernelSHAP {game.calls}')

Let us run each estimator at each budget with 30 seeds: 5 estimators, 5 budgets, 150 runs per estimator. On a Colab CPU this takes about a minute.

In [ ]:
runs = {(name, budget): np.array([estimate(budget, seed) for seed in range(n_seeds)])
        for name, estimate in estimators.items() for budget in budgets}
print({key: value.shape for key, value in list(runs.items())[:2]})

## Task 1. Bias and variance

We measure the error of an estimator at a given budget by the total mean squared error over the 10 features, averaged over seeds: $\mathrm{MSE} = \frac{1}{30}\sum_{s}\sum_i \big(\hat\phi_i^{(s)} - \phi_i\big)^2$. It splits into two parts:

$$\mathrm{MSE} = \underbrace{\sum_i \big(\bar\phi_i - \phi_i\big)^2}_{\text{bias}^2} + \underbrace{\sum_i \frac{1}{30}\sum_s \big(\hat\phi_i^{(s)} - \bar\phi_i\big)^2}_{\text{variance}},$$

where $\bar\phi_i$ is the mean of the estimates of feature $i$ over the 30 seeds. Build a table `summary` indexed by (estimator, budget) with the columns `MSE`, `bias2` and `variance`. Check that `MSE` equals the sum of the other two columns.

Keep one thing in mind: even for an unbiased estimator, `bias2` computed from 30 seeds is not zero. The mean of 30 estimates is random itself, and its squared deviation equals the variance divided by 30 on average. So bias is visible only if `bias2` is larger than `variance / 30`; add this column to the table too.

In [ ]:
rows = []
for (name, budget), estimates in runs.items():
    mse = # Your code here
    bias2 = # Your code here
    variance = # Your code here
    rows.append({'estimator': name, 'budget': budget, 'MSE': mse, 'bias2': bias2, 'variance': variance,
                 'variance / 30': variance / n_seeds})
summary = pd.DataFrame(rows).set_index(['estimator', 'budget'])

print('max |MSE - bias2 - variance| =', np.abs(summary.MSE - summary.bias2 - summary.variance).max())
summary.round(3)

Let us plot MSE against the budget on log-log axes. A Monte Carlo error decreases as $1/\text{budget}$, which on such a plot is a straight line with slope $-1$; the gray dashed line shows this slope. At budget 1024 the estimators enumerate all coalitions, the error is zero, and this point has no place on a log axis.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
for name in estimators:
    mse = summary.loc[name].MSE.loc[budgets[:4]]                # at budget 1024 the error is zero and has no place on a log axis
    ax.plot(mse.index, mse.values, marker='o', label=name)
ax.plot(budgets[:4], [1000 * budgets[0] / b for b in budgets[:4]], '--', color='gray', label='slope -1')
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xticks(budgets[:4], budgets[:4])
ax.set_xlabel('budget: number of v(S) evaluations')
ax.set_ylabel('MSE against the exact values (30 seeds)')
ax.set_title('Kernel SHAP estimators, random forest on diabetes')
ax.legend()
plt.show()

**Self-check question.** Which estimator is the most accurate at budget 64, and what is its MSE? How many times larger is the MSE of `shapiq.UnbiasedKernelSHAP` with pairs than that of `shapiq.KernelSHAP` with pairs at budget 256? Is the bias of the original Kernel SHAP visible, and what happens at budget 1024?

<details><summary>Check yourself</summary>

At budget 64 the most accurate estimator is `shapiq.KernelSHAP` with pairs: MSE 1.272. At the same budget `shap.KernelExplainer` has 5.076 and `shapiq.KernelSHAP` without pairs 17.974. At budget 256 the MSE of `shapiq.UnbiasedKernelSHAP` with pairs is 127.451, and that of `shapiq.KernelSHAP` with pairs is 0.154, about 830 times smaller. With the same choice of pairing, the MSE of the unbiased estimator is 80 to 830 times larger than that of the original one at every budget from 64 to 512. Pairs of complementary coalitions lower the MSE of both: for `shapiq.KernelSHAP` at budget 256 from 2.359 to 0.154, for `shapiq.UnbiasedKernelSHAP` from 242.759 to 127.451.

The bias of the original Kernel SHAP is visible but small: for `shapiq.KernelSHAP` without pairs at budget 64, `bias2` = 0.866 against `variance / 30` = 0.570, while the variance itself is 17.108. For `shapiq.UnbiasedKernelSHAP`, `bias2` stays within the noise: at budget 256 it is 5.688 with `variance / 30` = 7.902. The absence of bias comes here with a variance 80 to 830 times larger. The MSE of `shap.KernelExplainer` falls 30-fold from 64 to 512 evaluations for an eightfold budget increase, faster than slope −1: the larger the budget, the more coalition sizes it enumerates completely. At budget 1024 the MSE of all five estimators is 0: `KernelExplainer` evaluates all 1022 coalitions except the empty and the full one, and the `shapiq` estimators at budget $2^{10}$ evaluate all coalitions and solve the regression exactly.

</details>

## Confidence intervals: `shapley-regression`

`shapiq` returns only point estimates. Confidence intervals and stopping on convergence are in the package by the authors of Unbiased KernelSHAP, [`shapley-regression`](https://github.com/iancovert/shapley-regression), imported as `shapreg`. In it, `shapley_unbiased.ShapleyRegression` is Unbiased KernelSHAP and `shapley.ShapleyRegression` is the original Kernel SHAP, both with pairs of complementary coalitions.

We build the game from the package parts: `removal.MarginalExtension(background, model.predict)` marginalizes the removed features over the same background, and `games.PredictionGame` fixes the explained row. The estimator draws coalitions in batches of `batch_size` pairs and after each batch recomputes the values and their standard errors $\hat\sigma_i$. It stops when the largest standard error falls below the fraction `thresh` of the range of the values:

$$\frac{\max_i \hat\sigma_i}{\max_i \hat\phi_i - \min_i \hat\phi_i} < \texttt{thresh}.$$

We take `thresh=0.05`: with a range of about 50 this means a standard error below 2.5. The package takes its randomness from `np.random`, so we set the seed with `np.random.seed`.

In [ ]:
prediction_game = games.PredictionGame(removal.MarginalExtension(background, model.predict), x)


def regression_with_intervals(method, seed):
    np.random.seed(seed)
    values, tracking = method(prediction_game, batch_size=32, detect_convergence=True, thresh=0.05,
                              paired_sampling=True, bar=False, return_all=True)
    return values, tracking


unbiased, unbiased_tracking = regression_with_intervals(shapley_unbiased.ShapleyRegression, seed=0)
original, original_tracking = regression_with_intervals(shapley.ShapleyRegression, seed=0)
print(f'Unbiased KernelSHAP stopped after {unbiased_tracking["iters"][-1]} evaluations of v(S), '
      f'Kernel SHAP after {original_tracking["iters"][-1]}')
pd.DataFrame({'exact': phi_exact,
              'Unbiased': unbiased.values, 'Unbiased, 1.96 sigma': 1.96 * unbiased.std,
              'Kernel SHAP': original.values, 'Kernel SHAP, 1.96 sigma': 1.96 * original.std},
             index=feature_names).round(2)

Let us see how the largest standard error of Unbiased KernelSHAP decreases as coalitions accumulate. The original Kernel SHAP from `shapley.ShapleyRegression` has no standard error along the way: it estimates the error from the spread of five independent sub-estimates of 4 batches each, so the first estimate appears only after 20 batches, 1280 evaluations of $v(S)$, and with our threshold it is also the last. On the plot it is a single point. The dashed line is the stopping threshold: `thresh` times the range of the exact values.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
largest_std = [np.max(std) for std in unbiased_tracking['std']]
ax.plot(unbiased_tracking['iters'][:len(largest_std)], largest_std, marker='.', label='shapley_unbiased: Unbiased KernelSHAP')
ax.scatter([original_tracking['iters'][-1]], [original.std.max()], color='tab:orange', zorder=3,
           label='shapley: Kernel SHAP, its only error estimate')
ax.axhline(0.05 * (phi_exact.max() - phi_exact.min()), color='gray', linestyle='--', label='stopping threshold')
ax.set_yscale('log')
ax.set_xlabel('values v(S) computed')
ax.set_ylabel('largest standard error')
ax.set_title('Convergence of shapley-regression, first diabetes test row')
ax.legend()
plt.show()

## Task 2. Do the intervals cover the exact value

The interval $\hat\phi_i \pm 1.96\,\hat\sigma_i$ promises to cover the exact $\phi_i$ in about 95% of cases. Let us check this promise for Unbiased KernelSHAP: run `regression_with_intervals(shapley_unbiased.ShapleyRegression, seed)` for seeds 0–29 and count how many of the $30 \times 10 = 300$ intervals covered the exact value (`covered`). Also save how many evaluations of $v(S)$ each run needed before stopping (`evaluations`).

In [ ]:
covered = 0
evaluations = []
for seed in range(n_seeds):
    # Your code here

print(f'{covered} of {n_seeds * n_players} intervals covered the exact value, share {covered / (n_seeds * n_players):.3f}')
print(f'v(S) evaluations before stopping: {np.mean(evaluations):.0f} on average, from {min(evaluations)} to {max(evaluations)}')

**Self-check question.** How many of the 300 intervals covered the exact value? How many evaluations of $v(S)$ did Unbiased KernelSHAP need before stopping on average, how many did the original Kernel SHAP need with seed 0, and how does this fit the table of Task 1?

<details><summary>Check yourself</summary>

287 of the 300 intervals covered the exact value, a share of 0.957, close to the promised 95%. Before stopping, Unbiased KernelSHAP needed 2114 evaluations of $v(S)$ on average (from 1728 to 2432), while the original Kernel SHAP with seed 0 needed 1280, and its intervals are 10 to 90 times narrower: ±0.16 against ±1.65 for `bmi`, ±0.06 against ±5.49 for `age`. This is the same picture as in Task 1: the variance of the unbiased estimate is larger, and it needs more coalitions to reach the same stopping threshold. Unbiased KernelSHAP is useful not for the accuracy of its point estimate but because the interval around an unbiased estimate covers the exact value with the promised frequency, and this was checked above.

</details>

## Where the estimators do not help: the choice of background

All the estimators above converge to one number: the exact Shapley value of the game with the chosen background. But the background is part of the definition of the game $v(S)$, and a different background gives a different game with different exact values. Let us compute exact values for backgrounds of 8, 64 and 256 random training rows, 5 different backgrounds of each size, and compare the spread of the exact values between backgrounds with the spread of the `shap.KernelExplainer` estimate at budget 512.

In [ ]:
background_spread = {}
for size in [8, 64, 256]:
    exact_by_background = []
    for seed in range(5):
        other_background = X_train[np.random.default_rng(100 + seed).choice(len(X_train), size, replace=False)]
        values = shapiq.ExactComputer(Game(model, x, other_background), n_players=n_players)('SV')
        exact_by_background.append([values[(i,)] for i in range(n_players)])
    exact_by_background = np.array(exact_by_background)
    background_spread[f'background of {size} rows'] = exact_by_background.std(axis=0)
    print(f'background of {size:>3} rows: bmi from {exact_by_background[:, 2].min():.2f} to {exact_by_background[:, 2].max():.2f}')

background_spread['KernelExplainer, budget 512'] = runs[('shap.KernelExplainer', 512)].std(axis=0)
pd.DataFrame(background_spread, index=feature_names).round(2)

**Self-check question.** Within what range does the exact value of `bmi` vary over five different backgrounds of 8 rows and of 256 rows? Which is larger for most features: the standard deviation of the exact values between backgrounds of 64 rows, or the standard deviation of the `KernelExplainer` estimate at budget 512?

<details><summary>Check yourself</summary>

Over five backgrounds of 8 rows the exact value of `bmi` varies from 32.46 to 55.49, over backgrounds of 256 rows from 46.85 to 47.49. Between backgrounds of 64 rows, the standard deviation of the exact values is larger than the standard deviation of the `KernelExplainer` estimate at budget 512 (0.12–0.15) for 9 of the 10 features: 1.61 against 0.12 for `bmi`, 1.17 against 0.12 for `s5`; it is smaller only for `sex` (0.09 against 0.12). So with a background of 64 rows, raising the budget beyond 512 reduces an error that is already smaller than the spread caused by the choice of background. Narrowing this spread takes a larger background, not more coalitions.

</details>

## Conclusion

Today you:
- computed the exact Shapley values of a random forest for one diabetes row by enumerating 1024 coalitions in `shapiq.ExactComputer` and checked them against the interventional `TreeExplainer` (a gap of at most 0.038 with values up to 48.2), while the path-dependent `TreeExplainer` computes a different game and differs by up to 4.16;
- compared five estimators at budgets from 64 to 1024 evaluations of $v(S)$: `shapiq.KernelSHAP` with pairs of complementary coalitions is the most accurate, the bias of the original Kernel SHAP is visible only at a small budget and is about 20 times smaller than its variance (`bias2` 0.866 against 17.108 at budget 64 without pairs), and the MSE of `shapiq.UnbiasedKernelSHAP` is 80 to 830 times larger than that of the original estimator with the same pairing;
- checked the intervals of Unbiased KernelSHAP from `shapley-regression`: 287 of 300 covered the exact value;
- saw that the choice of background changes the exact values themselves more than the error of `KernelExplainer` at budget 512.

On this dataset the conclusion is: for a point estimate the original Kernel SHAP with pairs is the better choice, and Unbiased KernelSHAP is worth taking when you need an interval with checked coverage, with a larger budget planned for it.

See you in the next steps!